In [3]:

import streamlit as st
import yaml
import os
import pandas as pd

ruta_datos = os.path.dirname(os.path.abspath(__file__))
YAML_DIR = os.path.join(ruta_datos, "YAML_data")


# -----------------------------
# CARGA DE YAML
# -----------------------------
def load_data(directory):
    data = []

    for filename in os.listdir(directory):
        if not (filename.endswith(".yaml") or filename.endswith(".yml")):
            continue

        path = os.path.join(directory, filename)

        with open(path, "r", encoding="utf-8") as f:
            content = yaml.safe_load(f) or {}

        edad_data = content.get("edad", {})
        pen_data = content.get("penetrancia", {})
        sev_data = content.get("gravedad", {})
        tr_data = content.get("efectividad_tratamiento", {})

        # Crear identificador OMIM principal
        omim_rel = content.get("OMIM_relacionados", [])
        omim_principal = omim_rel[0] if omim_rel else None

        data.append({
            "File": filename,

            # Datos generales
            "NBK": content.get("NBK"),
            "Archivo": content.get("Archivo"),
            "Enfermedad": content.get("ENFERMEDAD"),
            "Gen": content.get("HGNC"),
            "OMIM_enfermedad": omim_principal,
            "OMIM_relacionados": omim_rel,
            "Ultima_actualizacion": content.get("Ultima_actualizacion"),

            # Edad
            "Edad": edad_data.get("valor_raso"),
            "Edad_ref": edad_data.get("referencia"),
            "Edad_fiabilidad": edad_data.get("porcentaje_fiabilidad"),

            # Penetrancia
            "Penetrancia": pen_data.get("valor"),
            "Pen_referencia": pen_data.get("referencia"),
            "Pen_asintomaticos": pen_data.get("asintomaticos"),
            "Pen_ref_asint": pen_data.get("referencia_ asintomaticos"),
            "Pen_zigosidad": pen_data.get("zigosidad"),
            "Pen_ref_zigo": pen_data.get("referencia_zigosidad"),
            "Pen_mecanismo": pen_data.get("mecanismo_patogenicidad"),
            "Pen_funcional": pen_data.get("efecto_funcional"),
            "Pen_ref_funcional": pen_data.get("referencia_efecto_funcional"),
            "Pen_fiabilidad": pen_data.get("porcentaje_fiabilidad"),

            # Gravedad
            "Gravedad": sev_data.get("valor"),
            "Gravedad_ref": sev_data.get("referencia"),
            "Gravedad_fiabilidad": sev_data.get("porcentaje_fiabilidad"),

            # Tratamiento
            "Trat_existencia": tr_data.get("existencia"),
            "Trat_ref_exist": tr_data.get("referencia1") or tr_data.get("referencia"),
            "Trat_intervencion": tr_data.get("intervencion"),
            "Trat_ref_intervencion": tr_data.get("referencia2"),
            "Trat_mejora": tr_data.get("valor"),
            "Trat_ref_mejora": tr_data.get("referencia"),
            "Trat_fiabilidad": tr_data.get("porcentaje_fiabilidad"),
        })

    return data


# -----------------------------
# FILTRADO
# -----------------------------
def filter_data(data, age_group):

    filtered = []
    failed = []

    age_map = {
        "Recién nacido (0-2)": (0, 2),
        "Infancia temprana (2-6)": (2, 6),
        "Niñez inmediata (6-12)": (6, 12),
        "Adolescencia (12-18)": (12, 18),
        "Adultez (+18)": (18, 200)
    }

    low, high = age_map.get(age_group, (0, 200))

    for item in data:

        # ---------------- EDAD ----------------
        try:
            edad_f = float(item["Edad"])
            match = low <= edad_f < high
        except:
            match = False

        # ---------------- GRAVEDAD ----------------
        grav = str(item["Gravedad"] or "").lower()
        grav_ok = grav in ["grave", "moderada", "moderado"]

        # ---------------- PENETRANCIA ----------------
        # Si no hay valor numérico pero hay mecanismo LOF y homocigosis,
        # consideramos evidencia fuerte.
        try:
            pen_ok = float(item["Penetrancia"]) > 70
        except:
            mecanismo = str(item["Pen_mecanismo"] or "").lower()
            zigo = str(item["Pen_zigosidad"] or "").lower()

            pen_ok = (
                "loss" in mecanismo
                or "lof" in mecanismo
                or "homocig" in zigo
            )

        # ---------------- TRATAMIENTO ----------------
        trat_ok = bool(
            item["Trat_existencia"]
            and item["Trat_mejora"]
        )

        item["Gen_OMIM"] = f"{item['Gen']} – {item['OMIM_enfermedad']}"

        if match and grav_ok and pen_ok and trat_ok:
            filtered.append(item)
        else:
            failed.append(item)

    return filtered, failed


# -----------------------------
# MOSTRAR DETALLES
# -----------------------------
def display_gene_details(g):

    st.markdown(
        f"## GEN: {g['Gen']} – ENFERMEDAD: {g['Enfermedad']}"
    )

    # ---------- INFORMACIÓN GENERAL ----------
    with st.expander("Información general"):

        st.write(f"**NBK:** {g['NBK']}")
        st.write(f"**Archivo fuente:** {g['Archivo']}")
        st.write(f"**Última actualización:** {g['Ultima_actualizacion']}")

        st.write("**OMIM relacionados:**")
        st.write(g["OMIM_relacionados"])

    # ---------- EDAD ----------
    with st.expander("Edad"):

        st.write(f"**Edad de inicio:** {g['Edad']}")
        st.write(f"**Fiabilidad:** {g['Edad_fiabilidad']}%")
        st.write(f"**Referencia:** {g['Edad_ref']}")

    # ---------- PENETRANCIA ----------
    with st.expander("Penetrancia"):

        pen_df = pd.DataFrame({
            "Campo": [
                "Valor",
                "Referencia",
                "Asintomáticos",
                "Zigosidad",
                "Mecanismo",
                "Efecto funcional",
                "Fiabilidad"
            ],
            "Valor": [
                g['Penetrancia'],
                g['Pen_referencia'],
                g['Pen_asintomaticos'],
                g['Pen_zigosidad'],
                g['Pen_mecanismo'],
                g['Pen_funcional'],
                g['Pen_fiabilidad']
            ]
        })

        st.table(pen_df)

    # ---------- GRAVEDAD ----------
    with st.expander("Gravedad"):

        st.write(f"**Valor:** {g['Gravedad']}")
        st.write(f"**Fiabilidad:** {g['Gravedad_fiabilidad']}%")
        st.write(f"**Referencia:** {g['Gravedad_ref']}")

    # ---------- TRATAMIENTO ----------
    with st.expander("Tratamiento"):

        trat_df = pd.DataFrame({
            "Campo": [
                "Existe tratamiento",
                "Intervención",
                "Mejora",
                "Referencia existencia",
                "Referencia intervención",
                "Fiabilidad"
            ],
            "Valor": [
                g['Trat_existencia'],
                g['Trat_intervencion'],
                g['Trat_mejora'],
                g['Trat_ref_exist'],
                g['Trat_ref_intervencion'],
                g['Trat_fiabilidad']
            ]
        })

        st.table(trat_df)


# -----------------------------
# APP PRINCIPAL
# -----------------------------
def main():

    st.set_page_config(
        page_title="CRINGENES – Panel de genes",
        layout="wide"
    )

    st.title("CRINGENES – Panel de genes")

    # ---------- SIDEBAR ----------
    st.sidebar.header("Filtros")

    edad_sel = st.sidebar.selectbox(
        "Grupo de edad",
        [
            "Recién nacido (0-2)",
            "Infancia temprana (2-6)",
            "Niñez inmediata (6-12)",
            "Adolescencia (12-18)",
            "Adultez (+18)"
        ]
    )

    top_n = st.sidebar.radio(
        "Cantidad de genes",
        [3, 5, 10],
        index=2
    )

    # ---------- CARGA ----------
    data = load_data(YAML_DIR)

    filt, failed = filter_data(data, edad_sel)

    # ---------- RESULTADOS ----------
    st.subheader(f"Genes encontrados: {len(filt)}")

    if not filt:
        st.warning("No se encontraron genes con estos criterios")
        return

    df = pd.DataFrame(filt)

    # Ranking
    def rank_intervencion(row):

        interv = str(row.get("Trat_intervencion", "")).lower()
        mejora = bool(row.get("Trat_mejora"))

        if "pre" in interv and mejora:
            return 1

        if "post" in interv and mejora:
            return 2

        if row.get("Trat_existencia"):
            return 3

        return 4

    def rank_gravedad(val):

        val = str(val).lower()

        if val == "grave":
            return 1

        if val.startswith("moder"):
            return 2

        return 3

    df["Rank_Intervencion"] = df.apply(rank_intervencion, axis=1)
    df["Rank_Gravedad"] = df["Gravedad"].apply(rank_gravedad)

    df = df.sort_values(
        by=["Rank_Intervencion", "Rank_Gravedad"],
        ascending=[True, True]
    ).head(top_n)

    # Tabla principal
    st.dataframe(
        df[[
            "Gen",
            "Enfermedad",
            "Edad",
            "Gravedad",
            "Trat_existencia"
        ]],
        use_container_width=True
    )

    # Selector
    gen_sel = st.selectbox(
        "Selecciona un gen",
        df["Gen_OMIM"].unique()
    )

    g = df[df["Gen_OMIM"] == gen_sel].iloc[0]

    display_gene_details(g)


if __name__ == "__main__":
    main()




NameError: name '__file__' is not defined